In [1]:
# Fetch the data from different sources

# ICOS-Carbon-data- portal: https://github.com/ICOS-Carbon-Portal/data?tab=readme-ov-file#downloading-originals-programmatically

# Create an ICOS Carbon portal account: https://cpauth.icos-cp.eu/login/


In [2]:
import json
import os
import zipfile

import polars as pl
from icoscp_core.icos import bootstrap

In [3]:
token_file_path = "../tokens/cpauthToken_auth_conf.json"

In [ ]:
def get_icoscp_credentials(token_file_path):
    """Load the authentication token from the provided JSON file."""
    with open(token_file_path, "r") as f:
        credentials = json.load(f)
    return credentials


credentials = get_icoscp_credentials(token_file_path)

meta, data = bootstrap.fromCredentials(credentials["username"], credentials["password"])

In [5]:
# from icoscp_core.icos import meta

# fetches the list of known data types, including metadata associated with them
all_datatypes = meta.list_datatypes()

# data types with structured data access
previewable_datatypes = [dt for dt in all_datatypes if dt.has_data_access]

datatypes = [{"Description": dt.label, "uri": dt.uri} for dt in previewable_datatypes]

df = pl.DataFrame(datatypes)

df.write_csv("../data/intermediate/previewable_datatypes.csv")

df.head()

Description,uri
str,str
"""AirCore vertical profile level…","""http://meta.icos-cp.eu/resourc…"
"""Atmospheric CH4 product""","""http://meta.icos-cp.eu/resourc…"
"""Atmospheric CO product""","""http://meta.icos-cp.eu/resourc…"
"""Atmospheric CO2 product""","""http://meta.icos-cp.eu/resourc…"
"""Atmospheric GHG data product""","""http://meta.icos-cp.eu/resourc…"


In [ ]:
# from icoscp_core.icos import meta

# Ecosystem final quality (L2) product in ETC-Archive format - release 2025-1
collection_uri = "https://meta.icos-cp.eu/collections/1-HA2r4l5QUjAgQr5CCEfJe3"

""" 
The Level 2 Ecosystem collection consists of many zipped parts for 
individual stations and components. Examples (as shown on the collection page) 
include:
*_FLUXES_L2.zip — Flux measurements (GPP, NEE, latent/sensible heat)
*_METEO_L2.zip — Meteorological variables (radiation, temperature, humidity, 
    wind, etc.)
*_AUXDATA_L2.zip — Ancillary/biometric measurements

"""

collection_meta = meta.get_collection_meta(collection_uri)

members = collection_meta.members

In [7]:
collection_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"
collection_meta = meta.get_station_meta(collection_uri)
print(dir(collection_meta))

['__annotate_func__', '__annotations__', '__annotations_cache__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__replace__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'countryCode', 'coverage', 'funding', 'id', 'location', 'org', 'pictures', 'responsibleOrganization', 'specificInfo', 'staff']


In [42]:
# Extract the uri for meteorological data

meteorological_urls = []
for mem in members:
    if hasattr(mem, "name") and hasattr(mem, "res"):
        name = mem.name
        if isinstance(name, str) and "METEO" in name:
            meteorological_urls.append(mem.res)
            # print(mem.res, name)

# Extract uri's for fluxes
flux_urls = []
for mem in members:
    if hasattr(mem, "name") and hasattr(mem, "res"):
        name = mem.name
        if isinstance(name, str) and "FLUXES_L2" in name:
            flux_urls.append(mem.res)
            # print(mem.res, name)

# Extract uri's for ancillary measurememnts
aux_urls = []
for mem in members:
    if hasattr(mem, "name") and hasattr(mem, "res"):
        name = mem.name
        if isinstance(name, str) and "AUXDATA" in name:
            aux_urls.append(mem.res)
            print(mem.res, name)

https://meta.icos-cp.eu/objects/MM03ZcoXByiQxNVDdaLQpM5z ICOSETC_BE-Bra_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/GTzxWow0u-RMUyykrPBvz5S1 ICOSETC_BE-Dor_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/Jtx_PJ1BGsl4o_o5oxtzyDd3 ICOSETC_BE-Lon_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/OxCaJl5bOkt-dWtoTzpYSPTd ICOSETC_BE-Maa_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/nsk8uCwLjbfh84hFlqZFw6AK ICOSETC_BE-Vie_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/DYbIBsXgz_3i17ZQctm80T2a ICOSETC_CH-Dav_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/R3l0tIWfq2jdXzojN4po2PkX ICOSETC_CZ-BK1_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/DSanpeYslisBVe8CF0HfY6bK ICOSETC_CZ-Lnz_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/B2JP5gN7RMNiyO7dKX5Yiaag ICOSETC_DE-Geb_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/04LgrLThcOIc6oI1bBCC9x1V ICOSETC_DE-HoH_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/wOQuQd4XKFiBvN4PT9WpHZYT ICOSETC_DE-RuS_AUXDATA_L2.zip
https://meta.icos-cp.eu/objects/LCGhDA_nG16

In [41]:
n20_urls = []
for mem in members:
    if hasattr(mem, "name") and hasattr(mem, "res"):
        name = mem.name
        if isinstance(name, str) and "N2O" in name:
            n20_urls.append(mem.res)
            print(mem.res, name)

In [10]:
from icoscp_core.icos import ATMO_STATION, meta

# fetch lists of stations, with basic metadata
icos_stations = meta.list_stations()
atmo_stations = meta.list_stations(ATMO_STATION)
all_known_stations = meta.list_stations(False)

# get detailed metadata for a station
htm_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"
htm_station_meta = meta.get_station_meta(htm_uri)

htm_station_meta

Station(org=Organization(self=UriResource(uri='http://meta.icos-cp.eu/resources/stations/AS_HTM', label='HTM', comments=[]), name='Hyltemossa', email=None, website=None, webpageDetails=None), id='HTM', location=Position(lat=56.0976, lon=13.4189, alt=115.0, label='HTM', uri=None), coverage=None, responsibleOrganization=Organization(self=UriResource(uri='http://meta.icos-cp.eu/resources/organizations/CEC', label='CEC', comments=[]), name='Centre for Environmental and Climate Science, Lund University', email=None, website='https://www.cec.lu.se', webpageDetails=None), pictures=['https://meta.icos-cp.eu/files/655_ISiY9jPRX2HHmCEW1_ux/Hyltemossa%202.JPG'], specificInfo=AtcStationSpecifics(wigosId='0-20008-0-HTM', theme=DataTheme(self=UriResource(uri='http://meta.icos-cp.eu/resources/themes/atmosphere', label='Atmospheric data', comments=[]), icon='https://static.icos-cp.eu/images/themes/atm.svg', markerIcon='https://static.icos-cp.eu/share/stations/icons/as.png'), stationClass='1', labeling

In [11]:
# Creating a table for ICOS stations
icos_stations = meta.list_stations()
station_data = []
schema = ["Id", "Label", "Name", "Country code", "Lat", "Lon"]

for station in icos_stations:
    station_data.append(
        {
            "Id": station.type_uri,
            "Label": station.label,
            "Name": station.name,
            "Country code": station.country_code,
            "Lat": station.lat,
            "Lon": station.lon,
        }
    )

station_data = pl.DataFrame(
    station_data,
    schema=schema,
)

station_data

Id,Label,Name,Country code,Lat,Lon
str,str,str,str,f64,f64
"""http://meta.icos-cp.eu/ontolog…","""El Arenosillo (ARN)""","""El Arenosillo""","""ES""",37.104,-6.734
"""http://meta.icos-cp.eu/ontolog…","""Innsbruck (AT-Inn)""","""Innsbruck""","""AT""",47.264095,11.385816
"""http://meta.icos-cp.eu/ontolog…","""Mieming (AT-Mmg)""","""Mieming""","""AT""",47.316563,10.970089
"""http://meta.icos-cp.eu/ontolog…","""Neustift (AT-Neu)""","""Neustift""","""AT""",47.116318,11.320229
"""http://meta.icos-cp.eu/ontolog…","""Puergschachen Moor (AT-PsM)""","""Puergschachen Moor""","""AT""",47.58145,14.347197
…,…,…,…,…,…
"""http://meta.icos-cp.eu/ontolog…","""Umhlabuyalingana (ZA-Uby)""","""Umhlabuyalingana""","""ZA""",-27.3952,32.5889
"""http://meta.icos-cp.eu/ontolog…","""Zeppelin (ZEP)""","""Zeppelin""","""NO""",78.9072,11.8867
"""http://meta.icos-cp.eu/ontolog…","""Zotino (ZOT)""","""Zotino""","""RU""",64.48,89.21


In [12]:
icos_stations

[StationLite(uri='http://meta.icos-cp.eu/resources/stations/AS_ARN', label='El Arenosillo (ARN)', comments=[], id='ARN', type_uri='http://meta.icos-cp.eu/ontologies/cpmeta/AS', name='El Arenosillo', country_code='ES', lat=37.104, lon=-6.734, elevation=42.0, geo_json=None),
 StationLite(uri='http://meta.icos-cp.eu/resources/stations/ES_AT-Inn_2', label='Innsbruck (AT-Inn)', comments=[], id='AT-Inn', type_uri='http://meta.icos-cp.eu/ontologies/cpmeta/ES', name='Innsbruck', country_code='AT', lat=47.264095, lon=11.385816, elevation=574.0, geo_json=None),
 StationLite(uri='http://meta.icos-cp.eu/resources/stations/ES_AT-Mmg', label='Mieming (AT-Mmg)', comments=[], id='AT-Mmg', type_uri='http://meta.icos-cp.eu/ontologies/cpmeta/ES', name='Mieming', country_code='AT', lat=47.316563, lon=10.970089, elevation=960.0, geo_json=None),
 StationLite(uri='http://meta.icos-cp.eu/resources/stations/ES_AT-Neu', label='Neustift (AT-Neu)', comments=[], id='AT-Neu', type_uri='http://meta.icos-cp.eu/ontolo

In [13]:
from icoscp_core.metaclient import SamplingHeightFilter, SizeFilter, TimeFilter

# list data objects with basic metadata
# a contrived, complicated example to demonstrate the possibilities
# all the arguments are optional
# see the Python help for the method for more details
filtered_atc_co2 = meta.list_data_objects(
    datatype=[
        "http://meta.icos-cp.eu/resources/cpmeta/atcCo2L2DataObject",
        "http://meta.icos-cp.eu/resources/cpmeta/atcCo2NrtGrowingDataObject",
    ],
    station="http://meta.icos-cp.eu/resources/stations/AS_GAT",
    filters=[
        TimeFilter("submTime", ">", "2016-07-01T12:00:00Z"),
        TimeFilter("submTime", "<", "2018-07-10T12:00:00Z"),
        SizeFilter(">", 5000),
        SamplingHeightFilter("=", 216),
    ],
    include_deprecated=True,
    order_by="fileName",
    # limit = 50
)

filtered_atc_co2

[DataObjectLite(uri='https://meta.icos-cp.eu/objects/yGh8S2hyBa3xnD0KKq4zp3Ul', filename='ICOS_ATC_L2_L2pre2018.1_GAT_216.0_489_CO2.zip', size_bytes=130398, datatype_uri='http://meta.icos-cp.eu/resources/cpmeta/atcCo2L2DataObject', station_uri='http://meta.icos-cp.eu/resources/stations/AS_GAT', sampling_height=216.0, submission_time=datetime.datetime(2018, 5, 23, 8, 12, 41, 724000, tzinfo=datetime.timezone.utc), time_start=datetime.datetime(2017, 4, 11, 0, 0, tzinfo=datetime.timezone.utc), time_end=datetime.datetime(2017, 12, 31, 23, 0, tzinfo=datetime.timezone.utc))]

In [14]:
from icoscp_core.icos import meta

station_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"

# List ALL data objects associated with this station
dobjs = meta.list_data_objects(station=station_uri, limit=10000)

# Extract URIs
dobj_uris = [dobj.uri for dobj in dobjs]

fluxes_l2 = [d for d in dobjs if "FLUXES" in d.filename]

len(dobjs)

10000

In [15]:
dobjs = meta.list_data_objects(station=station_uri, limit=10000)
for d in dobjs:
    print(d.uri, d.filename)

https://meta.icos-cp.eu/objects/pWF4dPTeCkOtHzOC4r7E_jcx ICOS_ATC_NRT_HTM_2025-05-19_2026-03-03_70.0_1699-506_N2O.zip
https://meta.icos-cp.eu/objects/4qqbxxrG9F5t5ffQJGlMNuVp ICOS_ATC_NRT_HTM_2025-04-01_2026-03-03_70.0_1238_CO2.zip
https://meta.icos-cp.eu/objects/LMt0jdvHh27L5OrCmCpJliwe ICOS_ATC_NRT_HTM_2025-04-01_2026-03-03_70.0_1238_CH4.zip
https://meta.icos-cp.eu/objects/R4l3EVUQXvwt20mJNDUTkt1d ICOS_ATC_NRT_HTM_2025-04-01_2026-03-03_70.0_1238-1699-506_CO.zip
https://meta.icos-cp.eu/objects/MeQT9hDnFTyWUchL44ezPipi ICOS_ATC_NRT_HTM_2025-05-19_2026-03-03_30.0_1699-506_N2O.zip
https://meta.icos-cp.eu/objects/kC_4RMBhiCCPWP08B99EsyCA ICOS_ATC_NRT_HTM_2025-04-01_2026-03-03_30.0_1238_CO2.zip
https://meta.icos-cp.eu/objects/tYhMcQzPloss960OEd73PDFB ICOS_ATC_NRT_HTM_2025-04-01_2026-03-03_30.0_1238_CH4.zip
https://meta.icos-cp.eu/objects/K-5EzUF_f8iNUXzAGOa2f1cT ICOS_ATC_NRT_HTM_2025-04-01_2026-03-03_30.0_1238-1699-506_CO.zip
https://meta.icos-cp.eu/objects/RRpMkjeivgx6W4DGOSMshuec ICOS_AT

In [16]:
from icoscp_core.icos import station_class_lookup

htm_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"
htm_class = station_class_lookup()[htm_uri]

htm_class

'1'

In [17]:
htm_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"
htm_station_meta = meta.get_station_meta(htm_uri)

htm_station_meta.__dict__

{'org': Organization(self=UriResource(uri='http://meta.icos-cp.eu/resources/stations/AS_HTM', label='HTM', comments=[]), name='Hyltemossa', email=None, website=None, webpageDetails=None),
 'id': 'HTM',
 'location': Position(lat=56.0976, lon=13.4189, alt=115.0, label='HTM', uri=None),
 'coverage': None,
 'responsibleOrganization': Organization(self=UriResource(uri='http://meta.icos-cp.eu/resources/organizations/CEC', label='CEC', comments=[]), name='Centre for Environmental and Climate Science, Lund University', email=None, website='https://www.cec.lu.se', webpageDetails=None),
 'pictures': ['https://meta.icos-cp.eu/files/655_ISiY9jPRX2HHmCEW1_ux/Hyltemossa%202.JPG'],
 'specificInfo': AtcStationSpecifics(wigosId='0-20008-0-HTM', theme=DataTheme(self=UriResource(uri='http://meta.icos-cp.eu/resources/themes/atmosphere', label='Atmospheric data', comments=[]), icon='https://static.icos-cp.eu/images/themes/atm.svg', markerIcon='https://static.icos-cp.eu/share/stations/icons/as.png'), statio

In [18]:
from icoscp_core.icos import meta

station_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"

all_objects = []
offset = 0
batch_size = 1000  # fetch 1000 at a time

while True:
    batch = meta.list_data_objects(station=station_uri, limit=batch_size, offset=offset)
    if not batch:
        break
    all_objects.extend(batch)
    offset += batch_size
    # print(f"Fetched {len(all_objects)} objects so far...")

print(f"Total objects fetched: {len(all_objects)}")

Total objects fetched: 35202


In [19]:
all_objects[0].__dict__

{'uri': 'https://meta.icos-cp.eu/objects/pWF4dPTeCkOtHzOC4r7E_jcx',
 'filename': 'ICOS_ATC_NRT_HTM_2025-05-19_2026-03-03_70.0_1699-506_N2O.zip',
 'size_bytes': 43380,
 'datatype_uri': 'http://meta.icos-cp.eu/resources/cpmeta/atcN2oNrtGrowingDataObject',
 'station_uri': 'http://meta.icos-cp.eu/resources/stations/AS_HTM',
 'sampling_height': 70.0,
 'submission_time': datetime.datetime(2026, 3, 4, 11, 9, 17, 527000, tzinfo=datetime.timezone.utc),
 'time_start': datetime.datetime(2025, 5, 19, 16, 0, tzinfo=datetime.timezone.utc),
 'time_end': datetime.datetime(2026, 3, 3, 23, 0, tzinfo=datetime.timezone.utc)}

In [20]:
all_objects

[DataObjectLite(uri='https://meta.icos-cp.eu/objects/pWF4dPTeCkOtHzOC4r7E_jcx', filename='ICOS_ATC_NRT_HTM_2025-05-19_2026-03-03_70.0_1699-506_N2O.zip', size_bytes=43380, datatype_uri='http://meta.icos-cp.eu/resources/cpmeta/atcN2oNrtGrowingDataObject', station_uri='http://meta.icos-cp.eu/resources/stations/AS_HTM', sampling_height=70.0, submission_time=datetime.datetime(2026, 3, 4, 11, 9, 17, 527000, tzinfo=datetime.timezone.utc), time_start=datetime.datetime(2025, 5, 19, 16, 0, tzinfo=datetime.timezone.utc), time_end=datetime.datetime(2026, 3, 3, 23, 0, tzinfo=datetime.timezone.utc)),
 DataObjectLite(uri='https://meta.icos-cp.eu/objects/4qqbxxrG9F5t5ffQJGlMNuVp', filename='ICOS_ATC_NRT_HTM_2025-04-01_2026-03-03_70.0_1238_CO2.zip', size_bytes=113014, datatype_uri='http://meta.icos-cp.eu/resources/cpmeta/atcCo2NrtGrowingDataObject', station_uri='http://meta.icos-cp.eu/resources/stations/AS_HTM', sampling_height=70.0, submission_time=datetime.datetime(2026, 3, 4, 11, 9, 16, 524000, tzin

In [21]:
from icoscp_core.icos import meta

# List all data objects (returns DobjSpecLite objects)
all_objects = meta.list_data_objects()

# Extract URIs in the exact format you want
all_uris = [dobj.uri for dobj in all_objects]

# Inspect
len(all_uris), all_uris[:5]

(100,
 ['https://meta.icos-cp.eu/objects/Gyy707SMePiDDlFUCt8Du2Cq',
  'https://meta.icos-cp.eu/objects/uI-KaooRRgyJcWY_EM56UwI8',
  'https://meta.icos-cp.eu/objects/pdQm19rCVUsRdeaw1-GvO_QK',
  'https://meta.icos-cp.eu/objects/IFM7oxedUOnnMSdhU_zP90-c',
  'https://meta.icos-cp.eu/objects/YVlnWJlU7i2kUwX7japQ6W42'])

In [22]:
all_objects

[DataObjectLite(uri='https://meta.icos-cp.eu/objects/Gyy707SMePiDDlFUCt8Du2Cq', filename='n2o_isotopes_and_fluxes_agroscope_reckenholz_PARIS.zip', size_bytes=815408, datatype_uri='http://meta.icos-cp.eu/resources/cpmeta/genericDataArchiveL2', station_uri='http://meta.icos-cp.eu/resources/stations/Reckenholz', sampling_height=None, submission_time=datetime.datetime(2026, 5, 31, 23, 59, 59, tzinfo=datetime.timezone.utc), time_start=datetime.datetime(2024, 3, 5, 0, 0, tzinfo=datetime.timezone.utc), time_end=datetime.datetime(2024, 12, 11, 23, 59, 59, tzinfo=datetime.timezone.utc)),
 DataObjectLite(uri='https://meta.icos-cp.eu/objects/uI-KaooRRgyJcWY_EM56UwI8', filename='PARIS_WP3_F-GAS_obs_2025-07-04.zip', size_bytes=380017039, datatype_uri='http://meta.icos-cp.eu/resources/cpmeta/atmoMeasArchive', station_uri=None, sampling_height=None, submission_time=datetime.datetime(2026, 4, 1, 0, 0, tzinfo=datetime.timezone.utc), time_start=datetime.datetime(1987, 1, 23, 13, 57, tzinfo=datetime.time

In [23]:
all_objects[0].datatype_uri

'http://meta.icos-cp.eu/resources/cpmeta/genericDataArchiveL2'

In [24]:
dobj_uris = [
    "https://meta.icos-cp.eu/objects/E2MVHezJQReXShfzPtBVlVwS",
    "https://meta.icos-cp.eu/objects/udtdN1NfB3YNvclaEm5RpbCd",
]

# dobj_uris = ["https://meta.icos-cp.eu/objects/wFrOPLqm3CcOZpUmxfbPTGqo"]

folder_path = "../data/raw/ICOS/"

unsucessful_count = 0

for idx, dobj_uri in enumerate(meteorological_urls[0:1]):
    try:
        filename = data.save_to_folder(dobj_uri, folder_path)

        zip_file_path = os.path.join(folder_path, filename)
        print(filename, zip_file_path, zip_file_path[:-4])

        # Extract the ZIP file
        with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
            zip_ref.extractall(folder_path)

    except Exception as e:
        print(f"Error downloading file: {e} ----- {idx}----- {dobj_uri}")

ICOSETC_BE-Bra_METEOSENS_L2.zip ../data/raw/ICOS/ICOSETC_BE-Bra_METEOSENS_L2.zip ../data/raw/ICOS/ICOSETC_BE-Bra_METEOSENS_L2


In [25]:
meteorological_urls[0]

'https://meta.icos-cp.eu/objects/5PxQWHuF4f6dtvXiHU3G0hOF'

In [26]:
dobj = meta.get_dobj_meta("https://meta.icos-cp.eu/objects/hujSGCfmNIRdxtOcEvEJLxGM")
coll_info = dobj.parentCollections[0]
coll_label = coll_info.label
coll_2010 = meta.get_collection_meta(coll_info.uri)
coll_evapo_info = coll_2010.parentCollections[0]
coll_evapo = meta.get_collection_meta(coll_evapo_info.uri)
evapo_years = coll_evapo.members
top_coll_info = coll_evapo.parentCollections[0]
top_coll = meta.get_collection_meta(top_coll_info.uri)
top_subcols = top_coll.members

In [27]:
dobj.__dict__

{'hash': 'hujSGCfmNIRdxtOcEvEJLxGMQwyJG4kI7FSkotuXjqw',
 'accessUrl': 'https://data.icos-cp.eu/objects/hujSGCfmNIRdxtOcEvEJLxGM',
 'pid': '11676/hujSGCfmNIRdxtOcEvEJLxGM',
 'doi': '10.18160/5NZG-JMJE',
 'fileName': 'ET_2010_025_daily.nc',
 'size': 363256131,
 'submission': DataSubmission(submitter=Organization(self=UriResource(uri='http://meta.icos-cp.eu/resources/organizations/CP', label='CP', comments=[]), name='Carbon Portal', email=None, website=None, webpageDetails=None), start='2024-01-30T13:18:04.251759Z', stop='2024-01-30T13:25:19.875259Z'),
 'specification': DataObjectSpec(self=UriResource(uri='http://meta.icos-cp.eu/resources/cpmeta/bioModelSpatialFLUXCOM', label='Biosphere-Modeling spatial results (FLUXCOM)', comments=['Set up for FLUXCOM model outputs']), project=Project(self=UriResource(uri='http://meta.icos-cp.eu/resources/projects/misc', label='Miscellaneous', comments=['Various other data not associated with a specific project']), keywords=None), theme=DataTheme(self=Ur

In [28]:
dobj.parentCollections

[UriResource(uri='https://meta.icos-cp.eu/collections/_Sno9zc_18Oy3nwaKD-ZYUPO', label='FLUXCOM-X-BASE evapotranspiration for 2010', comments=[])]

In [29]:
dobj.parentCollections[0]

UriResource(uri='https://meta.icos-cp.eu/collections/_Sno9zc_18Oy3nwaKD-ZYUPO', label='FLUXCOM-X-BASE evapotranspiration for 2010', comments=[])

In [30]:
meta.get_collection_meta(dobj.parentCollections[0].uri)

StaticCollection(res='https://meta.icos-cp.eu/collections/_Sno9zc_18Oy3nwaKD-ZYUPO', hash='_Sno9zc_18Oy3nwaKD-ZYUPO', members=[PlainStaticObject(res='https://meta.icos-cp.eu/objects/hujSGCfmNIRdxtOcEvEJLxGM', hash='hujSGCfmNIRdxtOcEvEJLxGMQwyJG4kI7FSkotuXjqw', name='FLUXCOM-X daily evapotranspiration on global 0.25 degree grid for 2010'), PlainStaticObject(res='https://meta.icos-cp.eu/objects/8T05NeB2GVeZmvO_FfrDK2tQ', hash='8T05NeB2GVeZmvO_FfrDK2tQj_O0L91sv_KUJzDVUT4', name='FLUXCOM-X monthly diurnal cycle of evapotranspiration on global 0.25 degree grid for 2010'), PlainStaticObject(res='https://meta.icos-cp.eu/objects/9wY9BgrGDX2smpBxUd7kvTtR', hash='9wY9BgrGDX2smpBxUd7kvTtRFl2SjDebD3lqBVg43tQ', name='FLUXCOM-X monthly evapotranspiration on global 0.05 degree grid for 2010'), PlainStaticObject(res='https://meta.icos-cp.eu/objects/7SOLdmdcpCDcdJN_rw0XDNu6', hash='7SOLdmdcpCDcdJN_rw0XDNu67fzaSmbJiPNThNG9rZg', name='FLUXCOM-X monthly evapotranspiration on global 0.5 degree grid for 201

In [31]:
coll_2010 = meta.get_collection_meta(dobj.parentCollections[0].uri)

coll_2010.__dict__

{'res': 'https://meta.icos-cp.eu/collections/_Sno9zc_18Oy3nwaKD-ZYUPO',
 'hash': '_Sno9zc_18Oy3nwaKD-ZYUPO',
 'members': [PlainStaticObject(res='https://meta.icos-cp.eu/objects/hujSGCfmNIRdxtOcEvEJLxGM', hash='hujSGCfmNIRdxtOcEvEJLxGMQwyJG4kI7FSkotuXjqw', name='FLUXCOM-X daily evapotranspiration on global 0.25 degree grid for 2010'),
  PlainStaticObject(res='https://meta.icos-cp.eu/objects/8T05NeB2GVeZmvO_FfrDK2tQ', hash='8T05NeB2GVeZmvO_FfrDK2tQj_O0L91sv_KUJzDVUT4', name='FLUXCOM-X monthly diurnal cycle of evapotranspiration on global 0.25 degree grid for 2010'),
  PlainStaticObject(res='https://meta.icos-cp.eu/objects/9wY9BgrGDX2smpBxUd7kvTtR', hash='9wY9BgrGDX2smpBxUd7kvTtRFl2SjDebD3lqBVg43tQ', name='FLUXCOM-X monthly evapotranspiration on global 0.05 degree grid for 2010'),
  PlainStaticObject(res='https://meta.icos-cp.eu/objects/7SOLdmdcpCDcdJN_rw0XDNu6', hash='7SOLdmdcpCDcdJN_rw0XDNu67fzaSmbJiPNThNG9rZg', name='FLUXCOM-X monthly evapotranspiration on global 0.5 degree grid for 20

In [32]:
coll_evapo = meta.get_collection_meta(coll_2010.parentCollections[0].uri)

coll_evapo.__dict__

{'res': 'https://meta.icos-cp.eu/collections/_l85vWiIV81AifoxCkty50YI',
 'hash': '_l85vWiIV81AifoxCkty50YI',
 'members': [PlainStaticCollection(res='https://meta.icos-cp.eu/collections/6uqHBICxAdAl55NtTkbAYw-s', hash='6uqHBICxAdAl55NtTkbAYw-s', title='FLUXCOM-X-BASE evapotranspiration for 2001'),
  PlainStaticCollection(res='https://meta.icos-cp.eu/collections/MV_IZZgxhXu1EebCL4IP-h7B', hash='MV_IZZgxhXu1EebCL4IP-h7B', title='FLUXCOM-X-BASE evapotranspiration for 2002'),
  PlainStaticCollection(res='https://meta.icos-cp.eu/collections/ewacq5ehLE4Z5WvWIRzk1vFw', hash='ewacq5ehLE4Z5WvWIRzk1vFw', title='FLUXCOM-X-BASE evapotranspiration for 2003'),
  PlainStaticCollection(res='https://meta.icos-cp.eu/collections/BrmOIOtF3iGe7Y-ab0mdBY4x', hash='BrmOIOtF3iGe7Y-ab0mdBY4x', title='FLUXCOM-X-BASE evapotranspiration for 2004'),
  PlainStaticCollection(res='https://meta.icos-cp.eu/collections/M-BBo0WTio_NoQMwUUB8A600', hash='M-BBo0WTio_NoQMwUUB8A600', title='FLUXCOM-X-BASE evapotranspiration fo

In [33]:
evapo_01_21 = meta.get_collection_meta(coll_evapo.parentCollections[0].uri)

evapo_01_21.__dict__

{'res': 'https://meta.icos-cp.eu/collections/zfwf1Ak2I7OlziGDTX8Xl6_T',
 'hash': 'zfwf1Ak2I7OlziGDTX8Xl6_T',
 'members': [PlainStaticCollection(res='https://meta.icos-cp.eu/collections/_l85vWiIV81AifoxCkty50YI', hash='_l85vWiIV81AifoxCkty50YI', title='FLUXCOM-X-BASE evapotranspiration for 2001-2021'),
  PlainStaticCollection(res='https://meta.icos-cp.eu/collections/AYj7-lwcdCLnBXJDoscxQZou', hash='AYj7-lwcdCLnBXJDoscxQZou', title='FLUXCOM-X-BASE gross primary productivity for 2001-2021'),
  PlainStaticCollection(res='https://meta.icos-cp.eu/collections/X_-p994BqVSG6nWJkeK1ALjW', hash='X_-p994BqVSG6nWJkeK1ALjW', title='FLUXCOM-X-BASE net ecosystem exchange for 2001-2021'),
  PlainStaticCollection(res='https://meta.icos-cp.eu/collections/aCBG-AZIJtCCia7bhMYWin1-', hash='aCBG-AZIJtCCia7bhMYWin1-', title='FLUXCOM-X-BASE transpiration for 2001-2021')],
 'creator': Organization(self=UriResource(uri='http://meta.icos-cp.eu/resources/organizations/CP', label='CP', comments=[]), name='Carbon Po